To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your own computer, follow the installation instructions on our Github page [here](https://unsloth.ai/docs/get-started/install).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it

### News

Long-Context GRPO for reinforcement learning — train stably at massive sequence lengths. Fine-tune models with up to 7x more context length efficiently. [Read Blog](https://unsloth.ai/docs/new/grpo-long-context)

3× faster training with optimized sequence packing — higher throughput with no quality loss.[Read Blog](https://unsloth.ai/docs/new/3x-faster-training-packing)

500k context-length fine-tuning — push long-context models further with memory-efficient training. [Read Blog](https://unsloth.ai/docs/new/500k-context-length-fine-tuning)

Introducing FP8 precision training for faster RL inference. [Read Blog](https://unsloth.ai/docs/new/fp8-reinforcement-learning).

Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://unsloth.ai/docs/new/how-to-train-llms-with-unsloth-and-docker).

Visit our docs for all our [model uploads](https://unsloth.ai/docs/get-started/all-our-models) and [notebooks](https://unsloth.ai/docs/get-started/unsloth-notebooks).

### Installation

In [1]:

%%capture
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.26.2 unsloth unsloth_zoo
!uv pip install transformers==5.3.0 weave mergekit
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0

### Unsloth

We're also introducing how you can do `GSPO` inside of Unsloth as well!

The goal of this notebook is to make a vision language model solve maths problems via reinforcement learning given an image input like below:

<img src="https://raw.githubusercontent.com/lupantech/MathVista/main/assets/our_new_3_datasets.png" alt="Alt text" height="256">

In [5]:
import unsloth

from unsloth import FastVisionModel
import torch
max_seq_length = 16384 # Must be this long for VLMs
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastVisionModel.from_pretrained(

    model_name = "/home/hongchang/.cache/huggingface/hub/models--unsloth--Qwen3.5-9B/snapshots/2da3e7c54ea56d5e12f44d81f16a763234b4e832",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = False, # Enable vLLM fast inference
    #device_map = "auto",
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


==((====))==  Unsloth 2026.3.7: Fast Qwen3_5 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA RTX 6000 Ada Generation. Num GPUs = 2. Max memory: 47.382 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights: 100%|██████████| 760/760 [00:06<00:00, 123.49it/s]


In Unsloth, we share vLLM's weights directly, reducing VRAM usage by > 50%. vLLM also does not yet support LoRA on the vision layers, so we can only add them on the language layers. Vision GRPO still works though!

In [6]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # False if not finetuning vision layers
    finetune_language_layers   = True,  # False if not finetuning language layers
    finetune_attention_modules = True,  # False if not finetuning attention layers
    finetune_mlp_modules       = True,  # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


### Data Prep
<a name="Data"></a>

`AI4Math/MathVista` is a dataset that involves using images to solve logic and math problems.

For this notebook, we will only use math problems with numeric answers for simplicity.

In [7]:

import wandb
import trl
from datasets import Dataset
from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer
import json
from pathlib import Path
from PIL import Image

_SCRIPT_DIR  = Path.cwd()
_DATASET_PATH = _SCRIPT_DIR / "data_collection" / "dataset" / "dataset_clean.jsonl"
_IMAGES_DIR   = _SCRIPT_DIR / "data_collection" / "images"

We filter the dataset to keep only float or numeric answers:

In [23]:

def _load_local_dataset() -> Dataset:
    """
    加载本地 dataset_clean.jsonl，将 image_path 解析为 PIL.Image。
    image_path 为 'images/xxx.ext' 相对路径，或 null（无图片时用纯白占位图）。
    统一字段：image（PIL.Image）、answer（ground_truth）、
              prompt_text（原始 prompt 文本）、reference_guideline
    """
    records = []
    with open(_DATASET_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            # 加载图片
            rel_path = rec.get("image_path")
            if rel_path:
                abs_path = _IMAGES_DIR / Path(rel_path).name
                try:
                    img = Image.open(abs_path).convert("RGB")
                except Exception:
                    img = Image.new("RGB", (512, 512), color=(255, 255, 255))
            else:
                img = Image.new("RGB", (512, 512), color=(255, 255, 255))
            records.append({
                "decoded_image":       img,
                "answer":              rec.get("ground_truth", ""),
                "prompt_text":         rec.get("prompt", ""),
                "reference_guideline": rec.get("reference_guideline", ""),
            })

    return Dataset.from_list(records)

dataset = _load_local_dataset()

We also resize the images to be 512 by 512 pixels to make the images managable in context length. We also convert them to RGB so they are compatible for training!

In [6]:
# Resize to (512, 512)
def resize_images(example):
    image = example["decoded_image"]
    image = image.resize((512, 512))
    example["decoded_image"] = image
    return example
dataset = dataset.map(resize_images)

# Then convert to RGB
def convert_to_rgb(example):
    image = example["decoded_image"]
    if image.mode != "RGB":
        image = image.convert("RGB")
    example["decoded_image"] = image
    return example
dataset = dataset.map(convert_to_rgb)

Map: 100%|██████████| 20128/20128 [04:56<00:00, 67.99 examples/s] 


We then create the conversational template that is needed to collate the dataset for RL:

In [22]:
# Define the delimiter variables for clarity and easy modification
REASONING_START = "<REASONING>"
REASONING_END = "</REASONING>"
SOLUTION_START = "<SOLUTION>"
SOLUTION_END = "</SOLUTION>"

def make_conversation(example):
    text_content = (
        f"{example['prompt_text']} "
        f"请先在 {REASONING_START} 和 {REASONING_END} 之间写出你的推理过程，"
        f"然后在 {SOLUTION_START} 和 {SOLUTION_END} 之间给出最终答案。"
    )

    prompt = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": text_content},
            ],
        },
    ]
    return {
        "prompt":              prompt,
        "image":               example["decoded_image"],
        "answer":              example["answer"],
        "reference_guideline": example["reference_guideline"],
    }

train_dataset = dataset.map(
    make_conversation,
    remove_columns=["decoded_image", "prompt_text"],  # 清理原始列，保留 prompt/image/answer/reference_guideline
)

NameError: name 'dataset' is not defined

Now let's apply the chat template across the entire dataset:

## Reward functions

We now define some basic formatting rewards functions to see if reasoning starts and ends, and also another to see if the answers were written correctly.

We also try to fix the `addCriterion` issue as described in our [blog post](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks)

In [ ]:
!pip install openai

In [11]:

# Reward functions
import re
import json
from openai import OpenAI

# vLLM 裁判模型客户端（Qwen3.5-9B，本地 8000 端口）
_JUDGE_CLIENT     = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="EMPTY")
_JUDGE_MODEL      = "/home/hongchang/.cache/modelscope/hub/models/Qwen/Qwen3.5-9B"
# 结构化返回 schema：score(0~3 整数) + reason(字符串)
_JUDGE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "score":  {"type": "integer", "minimum": 0, "maximum": 3},
        "reason": {"type": "string"}
    },
    "required": ["score", "reason"]
}


def _judge_single(completion: str, answer: str, reference_guideline: str) -> float:
    """
    调用裁判大模型对单条回答打分，返回归一化到 [0, 3] 的浮点分数。
    失败时返回 0.0。
    """
    system_prompt = (
        "你是一位严格的评分裁判。根据以下评分细则，对模型回答进行评分。\n"
        "评分细则：\n"
        f"{reference_guideline}\n\n"
        "评分规则：\n"
        "- 3分：推理过程正确且完整，最终答案与参考答案一致。\n"
        "- 2分：推理过程基本正确，最终答案与参考答案接近或部分正确。\n"
        "- 1分：推理过程有明显错误，但最终答案方向正确。\n"
        "- 0分：推理过程错误，最终答案错误或缺失。\n"
        "请以 JSON 格式返回：{\"score\": <0-3的整数>, \"reason\": \"<简短理由>\"}"
    )
    user_prompt = (
        f"参考答案：{answer}\n\n"
        f"模型回答：\n{completion}"
    )
    try:
        resp = _JUDGE_CLIENT.chat.completions.create(
            model=_JUDGE_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.0,
            max_tokens=256,
            extra_body={
                "guided_json": _JUDGE_JSON_SCHEMA,
            },
        )
        raw = resp.choices[0].message.content.strip()
        result = json.loads(raw)
        return float(result.get("score", 0))
    except Exception as e:
        print(f"[judge] 调用失败：{e}")
        return 0.0


def formatting_reward_func(completions, **kwargs) -> list[float]:
    """
    格式奖励：检查 <REASONING>...</REASONING> 和 <SOLUTION>...</SOLUTION> 是否各出现一次。
    同时惩罚 addCriterion / 大量换行等异常输出。
    """
    thinking_pattern = rf'{re.escape(REASONING_START)}(.*?){re.escape(REASONING_END)}'
    answer_pattern   = rf'{re.escape(SOLUTION_START)}(.*?){re.escape(SOLUTION_END)}'

    scores = []
    for completion in completions:
        if isinstance(completion, list):
            completion = completion[0]["content"] if completion else ""
        score = 0.0
        if re.findall(thinking_pattern, completion, re.DOTALL).__len__() == 1:
            score += 1.0
        if re.findall(answer_pattern, completion, re.DOTALL).__len__() == 1:
            score += 1.0
        # 惩罚 addCriterion / 换行泛滥（Qwen 已知问题）
        if len(completion) > 0:
            cleaned = completion.replace("addCriterion", "").replace("\n", "")
            if (len(completion) - len(cleaned)) / len(completion) >= 0.5:
                score -= 2.0
        scores.append(score)
    return scores


def correctness_reward_func(
    prompts, completions, answer, reference_guideline=None, **kwargs
) -> list[float]:
    """
    正确性奖励：调用 vLLM 部署的 Qwen3.5-9B 裁判模型，
    依据 reference_guideline 对每条回答打分（0~3），归一化到 [0, 1] 后返回。
    """
    completions = [
        (c[0]["content"] if c else "") if isinstance(c, list) else c
        for c in completions
    ]
    # reference_guideline 由 GRPOTrainer 从 dataset 列透传，可能是列表
    guidelines = reference_guideline if reference_guideline is not None else [""] * len(completions)

    scores = []
    for i, (completion, ans, guideline) in enumerate(zip(completions, answer, guidelines)):
        score = _judge_single(completion, ans, guideline)
        # 归一化：裁判满分 3 → 奖励 3.0（与 formatting 量级对齐）
        scores.append(score)
        print(
            f"[judge {i}] score={score:.1f} | "
            f"answer={ans!r} | "
            f"completion_head={completion[:80]!r}"
        )
    return scores


In [18]:
import json
from openai import OpenAI

# 假设你已经定义了 _JUDGE_CLIENT, _JUDGE_MODEL 和 _JUDGE_JSON_SCHEMA

def test_judge_connection():
    print(f"🚀 正在尝试连接本地裁判模型: {_JUDGE_MODEL} ...")
    
    # 模拟一个典型的 GRPO 奖励函数评价场景
    test_prompt = (
        "请评价以下预测答案是否符合参考指南。\n"
        "参考指南: 必须提到储氢材料的催化剂作用。\n"
        "预测答案: MgH2 的储氢性能通过加入 Ni@CeO2 催化剂得到了显著提升，降低了脱氢温度。"
    )

    try:
        # 使用 vLLM 专有的 guided_json 参数确保格式强制对齐
        response = _JUDGE_CLIENT.chat.completions.create(
            model=_JUDGE_MODEL,
            messages=[
                {"role": "system", "content": "你是一个严格的评分裁判，必须按照提供的 JSON 格式返回评分结果。"},
                {"role": "user", "content": test_prompt}
            ],
            extra_body={
                "guided_json": _JUDGE_JSON_SCHEMA  # 强制 vLLM 遵循你的 Schema
            },
            temperature=0.0  # 测试连通性通常使用 0 温度以获得稳定结果
        )

        # 解析返回内容
        content = response.choices[0].message.content
        print(f"\n📡 模型原始返回内容:\n{content}")

        result = json.loads(content)

        # 验证字段是否存在
        if "score" in result and "reason" in result:
            score = result["score"]
            reason = result["reason"]
            print("\n✅ 连通性测试通过！")
            print(f"📊 评分结果: {score}/3")
            print(f"📝 理由: {reason}")
        else:
            print("\n⚠️ 模型返回了 JSON 但缺少必要字段。")

    except ConnectionError:
        print("\n❌ 无法连接到服务器。请检查 vLLM 是否已在 8000 端口启动。")
    except Exception as e:
        print(f"\n❌ 测试过程中发生错误:\n{str(e)}")

if __name__ == "__main__":
    test_judge_connection()


🚀 正在尝试连接本地裁判模型: /home/hongchang/.cache/modelscope/hub/models/Qwen/Qwen3.5-9B ...

📡 模型原始返回内容:


{
  "score": 1,
  "reason": "预测答案明确提及了催化剂（Ni@CeO2）及其对储氢材料（MgH2）性能的提升作用，符合参考指南中必须提到储氢材料的催化剂作用的要求。"
}

✅ 连通性测试通过！
📊 评分结果: 1/3
📝 理由: 预测答案明确提及了催化剂（Ni@CeO2）及其对储氢材料（MgH2）性能的提升作用，符合参考指南中必须提到储氢材料的催化剂作用的要求。


In [24]:
import os

# 定义数据集保存的目录路径
_SAVE_DIR = _SCRIPT_DIR / "data_collection" / "processed_dataset"

# 如果目录不存在，可以自动创建（save_to_disk 通常会自动创建，但显式声明更安全）
_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 将 Dataset 保存到本地磁盘
train_dataset.save_to_disk(str(_SAVE_DIR))

print(f"✅ 数据集已成功保存至: {_SAVE_DIR}")

PermissionError: Tried to overwrite /home/hongchang/HydrogenChat_qwen3.5/Hydrogen_Chat/Auto_GRPO/data_collection/processed_dataset but a dataset can't overwrite itself.

In [15]:
from datasets import load_from_disk
from pathlib import Path

# 定义刚才保存的数据集路径
_SCRIPT_DIR = Path.cwd()
_LOAD_DIR = _SCRIPT_DIR / "data_collection" / "processed_dataset"

# 从本地磁盘加载 Dataset
train_dataset = load_from_disk(str(_LOAD_DIR))

print(f"✅ 数据集加载成功！共包含 {len(loaded_train_dataset)} 条数据。")

# 打印一下数据集的特征（features）以验证结构是否正确
print("\n数据集特征:")
print(loaded_train_dataset.features)

# 验证第一条数据的类型（例如确认 image 是否被正确识别为 PIL.Image）
if len(loaded_train_dataset) > 0:
    first_example = loaded_train_dataset[0]
    print(f"\n第一张图片类型: {type(first_example['image'])}")
    print(f"第一条数据的 prompt 结构: {first_example['prompt']}")


✅ 数据集加载成功！共包含 20128 条数据。

数据集特征:
{'answer': Value('string'), 'reference_guideline': Value('string'), 'prompt': List({'content': List({'text': Value('string'), 'type': Value('string')}), 'role': Value('string')}), 'image': Image(mode=None, decode=True)}

第一张图片类型: <class 'PIL.PngImagePlugin.PngImageFile'>
第一条数据的 prompt 结构: [{'content': [{'text': None, 'type': 'image'}, {'text': '什么因素促进了MgH2储氢性能的提升？ 请先在 <REASONING> 和 </REASONING> 之间写出你的推理过程，然后在 <SOLUTION> 和 </SOLUTION> 之间给出最终答案。', 'type': 'text'}], 'role': 'user'}]


Here is the first example prompt in the dataset

In [16]:
train_dataset[0]["prompt"]

[{'content': [{'text': None, 'type': 'image'},
   {'text': '什么因素促进了MgH2储氢性能的提升？ 请先在 <REASONING> 和 </REASONING> 之间写出你的推理过程，然后在 <SOLUTION> 和 </SOLUTION> 之间给出最终答案。',
    'type': 'text'}],
  'role': 'user'}]

In [17]:
train_dataset[100]["prompt"]

[{'content': [{'text': None, 'type': 'image'},
   {'text': 'Ni@CeO2对MgH2脱氢性能有什么影响？ 请先在 <REASONING> 和 </REASONING> 之间写出你的推理过程，然后在 <SOLUTION> 和 </SOLUTION> 之间给出最终答案。',
    'type': 'text'}],
  'role': 'user'}]

<a name="Inference"></a>
### Inference
Now let's try the model on the hundredth sample of the train dataset without training.

In [14]:
image = train_dataset[100]["image"]
prompt = train_dataset[100]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

KeyboardInterrupt: 

<a name="Train"></a>
### Train the model

Now set up the `GRPO` Trainer and all configurations! Note we actually enable `GSPO` as well!

tensorboard --logdir=outputs

In [25]:
from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    learning_rate = 5e-6,
    adam_beta1 = 0.9,
    adam_beta2 = 0.99,
    weight_decay = 0.1,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    log_completions = False,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 2, # Decrease if out of memory
    max_prompt_length = 1024,
    max_completion_length = 4096,
    # num_train_epochs = 0.5, # Set to 1 for a full training run
    max_steps = 60,
    save_steps = 60,
    max_grad_norm = 0.1,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",
    # Below enables GSPO:
    importance_sampling_level = "sequence",
    mask_truncated_completions = False,
    loss_type = "dr_grpo",
    bf16 = True,  
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |

During inference, you might encounter `addCriterion` or some weird gibberish outputs. Please read our [blog post](https://unsloth.ai/docs/new/vision-reinforcement-learning-vlm-rl#qwen-2.5-vl-vision-rl-issues-and-quirks) on why this occurs. It seems to be an inherent thing inside of the model, and we can ignore this.

In [26]:
trainer = GRPOTrainer(
    model = model,
    args = training_args,
    # Pass the processor to handle multimodal inputs
    processing_class = tokenizer,
    reward_funcs = [
        formatting_reward_func,
        correctness_reward_func,
    ],
    train_dataset = train_dataset,
)

trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,128 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 1 x 1) = 2
 "-____-"     Trainable parameters = 43,278,336 of 9,453,092,080 (0.46% trained)


[judge] 调用失败：'NoneType' object has no attribute 'strip'
[judge 0] score=0.0 | answer='2020年06月06日' | completion_head='<REASONING>\n提供的图像完全空白，没有任何可见的内容、文字、图表、数据或视觉信息。因此，无法从中提取与“论文答辩日期”相关的任何信息。在缺乏原始数据来'
[judge] 调用失败：'NoneType' object has no attribute 'strip'
[judge 1] score=0.0 | answer='2020年06月06日' | completion_head='<REASONING>\n由于提供的图像是纯白色，没有任何可识别的文本、图形、或文档内容，因此无法从中提取任何关于“论文答辩日期”的信息。\n在缺乏图像内容或对应文'
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / formatting_reward_func / mean,rewards / formatting_reward_func / std,rewards / correctness_reward_func / mean,rewards / correctness_reward_func / std
1,0.000000,2.000000,0.000000,101.000000,97.000000,105.000000,0.000000,101.000000,97.000000,105.000000,0.000037,2.000000,0.000000,0.000000,0.000000
2,0.000000,2.000000,0.000000,326.500000,274.000000,379.000000,0.000000,326.500000,274.000000,379.000000,0.000013,2.000000,0.000000,0.000000,0.000000
3,0.000000,2.000000,0.000000,693.500000,501.000000,886.000000,0.000000,693.500000,501.000000,886.000000,0.000012,2.000000,0.000000,0.000000,0.000000
4,0.000000,2.000000,0.000000,251.500000,167.000000,336.000000,0.000000,251.500000,167.000000,336.000000,0.000017,2.000000,0.000000,0.000000,0.000000
5,-0.001208,1.500000,0.707107,267.000000,260.000000,274.000000,0.000000,267.000000,260.000000,274.000000,0.000014,1.500000,0.707107,0.000000,0.000000
6,0.002848,1.500000,0.707107,550.500000,534.000000,567.000000,0.000000,550.500000,534.000000,567.000000,0.000014,1.500000,0.707107,0.000000,0.000000
7,-0.015103,1.500000,0.707107,253.500000,166.000000,341.000000,0.000000,253.500000,166.000000,341.000000,0.000023,1.500000,0.707107,0.000000,0.000000
8,0.000000,2.000000,0.000000,229.000000,209.000000,249.000000,0.000000,229.000000,209.000000,249.000000,0.000030,2.000000,0.000000,0.000000,0.000000
9,0.000000,2.000000,0.000000,596.500000,437.000000,756.000000,0.000000,596.500000,437.000000,756.000000,0.000020,2.000000,0.000000,0.000000,0.000000
10,0.000000,2.000000,0.000000,625.000000,367.000000,883.000000,0.000000,625.000000,367.000000,883.000000,0.000036,2.000000,0.000000,0.000000,0.000000


[judge] 调用失败：'NoneType' object has no attribute 'strip'
[judge 0] score=0.0 | answer='氯仿、丙酮、乙醇' | completion_head='<REASONING>\nFTO（氟掺杂二氧化钛）导电玻璃基片在制备太阳能电池或其他光电器件时，表面会吸附有机污染物、金属离子或残留物。为了获得洁净的表面，通常使'
[judge] 调用失败：'NoneType' object has no attribute 'strip'
[judge 1] score=0.0 | answer='氯仿、丙酮、乙醇' | completion_head='<REASONING>\nFTO（氟掺杂氧化锡）导电玻璃基片在物理气相沉积（PVD）等工艺前需要进行清洗，以去除表面的有机污染物、金属离子和颗粒。常见的清洗流程通'
[judge] 调用失败：'NoneType' object has no attribute 'strip'
[judge 0] score=0.0 | answer='不会改变。' | completion_head='<REASONING>\n1. MgH₂是一种储氢材料，其分解反应为：MgH₂ → Mg + H₂，该反应是吸热的，标准焓变 ΔH° ≈ +75 kJ/mol。其'
[judge] 调用失败：'NoneType' object has no attribute 'strip'
[judge 1] score=0.0 | answer='不会改变。' | completion_head='<REASONING>\n1.  **分析物质的化学性质**：\n    *   **MgH<sub>2</sub> (氢化镁)**：这是一种金属氢化物，主要作为一'
[judge] 调用失败：'NoneType' object has no attribute 'strip'
[judge 0] score=0.0 | answer='减小为原来的1/3' | completion_head='<REASONING>\n用户的问题是关于‘交错驱动控制技术’对‘变换器输出电流纹波’的影响。虽然提供的图像是完全空白的（无任何文字或图表），但问题明确提到‘根据'
[judge] 调用失败：'

TrainOutput(global_step=60, training_loss=-0.0016886683479452676, metrics={'train_runtime': 4272.9863, 'train_samples_per_second': 0.028, 'train_steps_per_second': 0.014, 'total_flos': 0.0, 'train_loss': -0.0016886683479452676})

<a name="Inference"></a>
### Inference

Let's run the model! You can modify the instruction and input.

In [ ]:
image = train_dataset[165]["image"]
prompt = train_dataset[165]["prompt"]

inputs = tokenizer(
    image,
    prompt,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024,
                   use_cache = True, temperature = 1.0, min_p = 0.1)

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, use Hugging Face’s `push_to_hub` for online saving, or `save_pretrained` for local storage.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("grpo_lora")  # Local saving
tokenizer.save_pretrained("grpo_lora")
# model.push_to_hub("your_name/grpo_lora", token = "...") # Online saving
# processor.push_to_hub("your_name/grpo_lora", token = "...") # Online saving

Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [30]:
# Merge to 16bit
if True: model.save_pretrained_merged("vllm_model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")

Detected local model directory: /home/hongchang/.cache/huggingface/hub/models--unsloth--Qwen3.5-9B/snapshots/2da3e7c54ea56d5e12f44d81f16a763234b4e832
Found HuggingFace hub cache directory: /home/hongchang/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  25%|██▌       | 1/4 [00:04<00:13,  4.35s/it]

Copied model.safetensors-00001-of-00004.safetensors from local model directory


Unsloth: Preparing safetensor model files:  50%|█████     | 2/4 [00:07<00:07,  3.62s/it]

Copied model.safetensors-00003-of-00004.safetensors from local model directory


...

Copied model.safetensors-00002-of-00004.safetensors from local model directory


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:11<00:00,  2.99s/it]


Copied model.safetensors-00004-of-00004.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:23<00:00, 35.89s/it]


Unsloth: Merge process complete. Saved to `/home/hongchang/HydrogenChat_qwen3.5/Hydrogen_Chat/Auto_GRPO/vllm_model`


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [29]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("ollama_model_8bit", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if True: model.save_pretrained_gguf("ollama_model_16bit", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Unsloth: Merging model weights to 16-bit format...
Detected local model directory: /home/hongchang/.cache/huggingface/hub/models--unsloth--Qwen3.5-9B/snapshots/2da3e7c54ea56d5e12f44d81f16a763234b4e832
Found HuggingFace hub cache directory: /home/hongchang/.cache/huggingface/hub


Unsloth: Preparing safetensor model files:  25%|██▌       | 1/4 [00:04<00:12,  4.21s/it]

Copied model.safetensors-00001-of-00004.safetensors from local model directory


Unsloth: Preparing safetensor model files:  50%|█████     | 2/4 [00:07<00:07,  3.72s/it]

Copied model.safetensors-00003-of-00004.safetensors from local model directory


...

Copied model.safetensors-00004-of-00004.safetensors from local model directory


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:20<00:00, 35.07s/it]


Unsloth: Merge process complete. Saved to `/home/hongchang/HydrogenChat_qwen3.5/Hydrogen_Chat/Auto_GRPO/ollama_model_8bit`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|ERROR]Unsloth: Error during download or introspection of original script: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Traceback (most recent call last):
  File "/home/hongchang/HydrogenChat_qwen3.5/Hydrogen_Chat/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/home/hongchang/HydrogenChat_qwen3.5/Hydrogen_Chat/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/home/hongchang/HydrogenChat_qwen3.5/Hydrogen_Chat/.venv/lib/python3.11/site-packages/urllib3/connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/http/client.py", line 1395, in getresponse
    response.begin()
  File "/usr/lib/pyt

RuntimeError: Unsloth: GGUF conversion failed: Failed during download/introspection of original script: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

Special Credits to [GAD-Cell](https://github.com/GAD-cell) for helping Unsloth create this notebook and bringing VLM GRPO into Unsloth!
Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other resources:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
4. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://unsloth.ai/docs/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme)